# DealerPulse — Exploratory Data Analysis

**Purpose:** lock down the metric definitions and surface the data's real story *before* any UI is written, so the app, the numbers, and `DECISIONS.md` all agree.

**Dataset:** `dealership_data.json` — 7 complete months (Jun–Dec 2025), 5 branches, 30 reps, 510 leads, 35 targets, 160 deliveries. All synthetic.

### Metric contract (the rules every calculation below obeys)
- **"Now" = dataset cutoff `2025-12-31`**, never the real wall-clock date. All staleness/aging is measured from here.
- **Delivery metrics** keyed on `delivery_date`; **lead-volume/source** on `created_at`.
- **Revenue** is attributed on delivery (a lead's `deal_value` counts as realized revenue only when `status == delivered`).
- **Funnel** reconstructed from `status_history` (a lead 'reached' a stage if that stage appears in its history), not from current `status`.
- **Targets** summed only over months inside the selected range; no partial-month proration.
- **Anomalies** are disclosed as 'Unknown', never silently dropped.


In [ ]:
import json, collections, statistics as st
from datetime import datetime, timezone

DATA_PATH = "../dealership_data.json"   # notebook lives in /analysis
d = json.load(open(DATA_PATH))
LEADS   = d["leads"]
BRANCH  = {b["id"]: b["name"] for b in d["branches"]}
CITY    = {b["id"]: b["city"] for b in d["branches"]}
REPS    = {r["id"]: r for r in d["sales_reps"]}
DELIV   = {x["lead_id"]: x for x in d["deliveries"]}
TARGETS = d["targets"]

CUTOFF = datetime(2025, 12, 31, tzinfo=timezone.utc)      # the analytical "now"
STAGES = ["new","contacted","test_drive","negotiation","order_placed","delivered"]
IDX    = {s:i for i,s in enumerate(STAGES)}
OPEN   = {"new","contacted","test_drive","negotiation","order_placed"}

def parse(s):
    return datetime.fromisoformat(s.replace("Z","+00:00"))

def reached_at(lead, status):
    """First timestamp at which a lead reached `status`, else None."""
    for h in lead["status_history"]:
        if h["status"] == status:
            return parse(h["timestamp"])
    return None

def stage_before_lost(lead):
    prev = "(unknown)"
    for h in lead["status_history"]:
        if h["status"] == "lost":
            return prev
        prev = h["status"]
    return prev

def cr(x):  # rupees -> crore string
    return f"Rs {x/1e7:.2f}cr"

print("Loaded:", len(LEADS), "leads |", len(d["branches"]), "branches |",
      len(REPS), "reps |", len(d["deliveries"]), "deliveries |", len(TARGETS), "targets")

: 

## 1. Shape & integrity — what can we trust?
Before analysis, validate the data and catalogue anomalies the UI must handle gracefully.

In [ ]:
status_dist = collections.Counter(l["status"] for l in LEADS)
print("Current-status distribution:", dict(status_dist))

# Anomaly A: leads marked 'lost' with no closing history event / no lost_reason
lost_no_reason = [l for l in LEADS if l["status"]=="lost" and not l.get("lost_reason")]
lost_no_hist   = [l for l in LEADS if l["status"]=="lost" and
                  (not l["status_history"] or l["status_history"][-1]["status"]!="lost")]
print(f"Anomaly A: 'lost' without lost_reason = {len(lost_no_reason)} ; "
      f"without final 'lost' history event = {len(lost_no_hist)}  -> render as 'Unknown'")

# Anomaly B: delivered leads missing a delivery record, and orphan deliveries
deliv_no_rec = [l for l in LEADS if l["status"]=="delivered" and l["id"] not in DELIV]
lead_ids = {l["id"] for l in LEADS}
orphan_deliv = [x for x in d["deliveries"] if x["lead_id"] not in lead_ids]
print(f"Anomaly B: delivered leads w/o delivery record = {len(deliv_no_rec)} ; "
      f"orphan deliveries = {len(orphan_deliv)}")

# deal_value coverage & stability
print("deal_value present on", sum(1 for l in LEADS if l.get('deal_value')), "of", len(LEADS), "leads")
print("status_history entries carry deal_value:",
      any('deal_value' in h for l in LEADS for h in l['status_history']))

## 2. The headline: systemic target shortfall
The dominant story is not a single weak branch — the whole group is running **far** below target.

In [ ]:
delivered = [l for l in LEADS if l["status"]=="delivered"]
total_target = sum(t["target_units"] for t in TARGETS)
print(f"Overall: {len(delivered)} units delivered vs {total_target} target = "
      f"{100*len(delivered)/total_target:.1f}% attainment  (systemic shortfall)")

# By month (deliveries keyed on delivery_date)
print("\nMonthly units delivered vs target:")
for m in sorted({t['month'] for t in TARGETS}):
    deld = sum(1 for x in d['deliveries'] if x['delivery_date'][:7]==m)
    tgt = sum(t['target_units'] for t in TARGETS if t['month']==m)
    print(f"  {m}  delivered={deld:3}  target={tgt:3}  attain={100*deld/tgt:4.1f}%")

# By branch: conversion + attainment
print("\nBy branch:")
by_b = collections.defaultdict(lambda:[0,0])
for l in LEADS:
    by_b[l['branch_id']][0]+=1
    if l['status']=='delivered': by_b[l['branch_id']][1]+=1
for bid in sorted(by_b):
    tot,dv = by_b[bid]
    tgt = sum(t['target_units'] for t in TARGETS if t['branch_id']==bid)
    print(f"  {BRANCH[bid]:18} ({CITY[bid]:9}) leads={tot:3} delivered={dv:3} "
          f"conv={100*dv/tot:4.1f}%  attain={100*dv/tgt:4.1f}%")

## 3. Conversion funnel & leak matrix
Rebuilt from `status_history`: of the leads that *reached* each stage, what share advanced to the next?

In [ ]:
reached = collections.Counter()
for l in LEADS:
    for s in set(h["status"] for h in l["status_history"]):
        if s in IDX: reached[s]+=1

print("Funnel (reached each stage) and stage-to-stage progression:")
for i,s in enumerate(STAGES):
    line = f"  {s:12} reached={reached[s]:3}"
    if i+1 < len(STAGES):
        nxt = STAGES[i+1]; adv = reached[nxt]
        line += f"  -> {nxt:12} advance={100*adv/reached[s]:3.0f}%  leak={100-100*adv/reached[s]:3.0f}%"
    print(line)

## 4. Where the money leaks — losses by stage, value, and reason
Attributing each loss to the stage it died at (and its `deal_value`) turns 'we lost 288 deals' into a targeted diagnosis.

In [ ]:
loss_cnt = collections.Counter(); loss_val = collections.Counter()
reason_by_stage = collections.defaultdict(collections.Counter)
for l in LEADS:
    if l["status"]=="lost":
        s = stage_before_lost(l)
        loss_cnt[s]+=1; loss_val[s]+=l["deal_value"]
        if l.get("lost_reason"):
            reason_by_stage[s][l["lost_reason"]]+=1

print("Losses by stage-before-lost (count & pipeline value):")
for s,_ in loss_cnt.most_common():
    print(f"  {s:12} losses={loss_cnt[s]:3}  value={cr(loss_val[s])}")

print("\nTop lost reasons by stage (maps each leak to a different fix):")
for s in ["new","contacted","test_drive","negotiation"]:
    if reason_by_stage[s]:
        print(f"  {s:12}", reason_by_stage[s].most_common(2))

print("\nOverall lost-reason ranking:")
allr = collections.Counter(l['lost_reason'] for l in LEADS
                           if l['status']=='lost' and l.get('lost_reason'))
for r,c in allr.most_common():
    print(f"  {r:32} {c}")

## 5. Pipeline velocity
Median time-in-stage (transition timestamps in `status_history`) and end-to-end sales cycle for delivered leads.

In [ ]:
durs = collections.defaultdict(list)
for l in LEADS:
    h = l["status_history"]
    for a,b in zip(h, h[1:]):
        if a["status"] in IDX and b["status"] in IDX and IDX[b["status"]]==IDX[a["status"]]+1:
            durs[a["status"]].append((parse(b["timestamp"])-parse(a["timestamp"])).total_seconds()/86400)

print("Median time-in-stage (days):")
for s in STAGES[:-1]:
    if durs[s]:
        print(f"  {s:12} median={st.median(durs[s]):4.1f}  mean={st.mean(durs[s]):4.1f}  n={len(durs[s])}")

cycle = [ (reached_at(l,'delivered')-parse(l['created_at'])).days
          for l in LEADS if l['status']=='delivered' and reached_at(l,'delivered') ]
cycle.sort()
print(f"\nSales cycle (created -> delivered): median={st.median(cycle)}d  "
      f"mean={st.mean(cycle):.1f}d  p90={cycle[int(0.9*len(cycle))]}d")

# Delivery SLA: delayed vs on-time
delays = [x for x in d['deliveries'] if x.get('delay_reason')]
ontime = [x for x in d['deliveries'] if not x.get('delay_reason')]
print(f"\nDeliveries: {len(delays)} delayed / {len(d['deliveries'])} "
      f"(avg days_to_deliver delayed={st.mean([x['days_to_deliver'] for x in delays]):.1f} "
      f"vs on-time={st.mean([x['days_to_deliver'] for x in ontime]):.1f})")
print("Top delay reasons:", collections.Counter(x['delay_reason'] for x in delays).most_common(4))

## 6. Channel & product quality (value-weighted)
Conversion alone undersells walk-ins; weighting by realized revenue-per-lead sharpens the spend-allocation story.

In [ ]:
sv = collections.defaultdict(lambda:[0,0,0.0])  # leads, delivered, revenue
for l in LEADS:
    sv[l['source']][0]+=1
    if l['status']=='delivered':
        sv[l['source']][1]+=1; sv[l['source']][2]+=l['deal_value']
print("Source quality:")
for s,(n,dc,rev) in sorted(sv.items(), key=lambda x:-x[1][2]/x[1][0]):
    print(f"  {s:14} leads={n:3} conv={100*dc/n:4.1f}%  rev/lead=Rs{rev/n/1e5:4.1f}L  total={cr(rev)}")

mv = collections.defaultdict(lambda:[0,0.0])
for l in LEADS:
    if l['status']=='delivered':
        mv[l['model_interested']][0]+=1; mv[l['model_interested']][1]+=l['deal_value']
print("\nModel revenue concentration (delivered):")
for m,(c,rev) in sorted(mv.items(), key=lambda x:-x[1][1]):
    print(f"  {m:22} units={c:3}  revenue={cr(rev)}")

## 7. The Action Center — deterministic, explainable alert rules
Every rule measures staleness from the **cutoff**, is fully explainable, and resolves to the exact affected leads with owner + value.

In [ ]:
def days_stale(l):
    return (CUTOFF - parse(l['last_activity_at'])).days

open_leads = [l for l in LEADS if l['status'] in OPEN]
cold = [l for l in open_leads if days_stale(l) >= 7]
op_stale = [l for l in open_leads if l['status']=='order_placed' and days_stale(l) >= 7]
overdue  = [l for l in open_leads if l.get('expected_close_date')
            and parse(l['expected_close_date']+'T00:00:00Z') < CUTOFF]

print(f"Open pipeline           : {len(open_leads):3} leads  value={cr(sum(l['deal_value'] for l in open_leads))}")
print(f"Cold (>=7d no activity) : {len(cold):3} leads  value={cr(sum(l['deal_value'] for l in cold))}")
print(f"Order placed but stale  : {len(op_stale):3} leads  value={cr(sum(l['deal_value'] for l in op_stale))}  <- highest urgency")
print(f"Past expected close date: {len(overdue):3} leads  value={cr(sum(l['deal_value'] for l in overdue))}")

# Simple at-risk score = value(cr) * stage_depth * log-ish age; illustrative ranking
def score(l):
    return (l['deal_value']/1e7) * (IDX.get(l['status'],0)+1) * max(days_stale(l),1)
top = sorted(open_leads, key=score, reverse=True)[:8]
print("\nTop at-risk leads (value x stage-depth x staleness):")
for l in top:
    print(f"  {l['id']} {l['customer_name']:18} {BRANCH[l['branch_id']]:16} "
          f"{l['status']:12} stale={days_stale(l):2}d  value={cr(l['deal_value'])}")

## 8. Rep & branch comparison (with a minimum-sample guardrail)
Lakeside is the clear operational outlier — but several of its reps have thin lead counts, so rankings are shown with sample size and a `>= 5 leads` floor to avoid unfair conclusions.

In [ ]:
rp = collections.defaultdict(lambda:[0,0,0.0])  # leads, delivered, revenue
for l in LEADS:
    rp[l['assigned_to']][0]+=1
    if l['status']=='delivered':
        rp[l['assigned_to']][1]+=1; rp[l['assigned_to']][2]+=l['deal_value']

rows = []
for rid,(n,dc,rev) in rp.items():
    if n >= 5:  # min-sample floor
        rows.append((100*dc/n, n, dc, rev, rid))
rows.sort()
print("Lowest-converting reps (>=5 leads):")
for conv,n,dc,rev,rid in rows[:5]:
    print(f"  {REPS[rid]['name']:18} {BRANCH[REPS[rid]['branch_id']]:16} "
          f"conv={conv:4.1f}%  leads={n:3}  revenue={cr(rev)}")
print("\nHighest-converting reps (>=5 leads):")
for conv,n,dc,rev,rid in rows[-5:][::-1]:
    print(f"  {REPS[rid]['name']:18} {BRANCH[REPS[rid]['branch_id']]:16} "
          f"conv={conv:4.1f}%  leads={n:3}  revenue={cr(rev)}")

## 9. Traps we tested and rejected
Documenting what we *didn't* build (and why) is a credibility signal in `DECISIONS.md`.

In [ ]:
# Trap 1: speed-to-lead (new -> contacted) does NOT predict conversion here
resp = []
for l in LEADS:
    tn, tc = reached_at(l,'new'), reached_at(l,'contacted')
    if tn and tc:
        resp.append(((tc-tn).total_seconds()/3600, l['status']=='delivered'))
print("Speed-to-lead vs conversion (noisy / no clean signal):")
for lo,hi,lbl in [(0,24,'<24h'),(24,72,'1-3d'),(72,1e9,'>3d')]:
    g=[d_ for h,d_ in resp if lo<=h<hi]
    if g: print(f"  {lbl:5} n={len(g):3} conv={100*sum(g)/len(g):4.1f}%")

# Trap 2: 'touch count' is circular -- delivered leads simply have all 6 stage entries
print("\n'Touch count' vs conversion (circular -- do NOT use as a predictor):")
tc = collections.defaultdict(lambda:[0,0])
for l in LEADS:
    n=len(l['status_history']); tc[n][0]+=1
    if l['status']=='delivered': tc[n][1]+=1
for n in sorted(tc):
    t,c=tc[n]; print(f"  {n} entries: n={t:3} conv={100*c/t:3.0f}%")

## Key takeaways for DECISIONS.md
1. **Systemic shortfall:** 160/1426 units = **11.2%** overall attainment; even the best month (Dec) hits only 23.9%. The dashboard must frame red numbers as a diagnosis, not a bug.
2. **Lakeside is the outlier:** 7.6% conversion vs 33–41% elsewhere; its reps fill the bottom of the leaderboard.
3. **Biggest leak is early & high-value:** 114 leads lost at `new` = **~Rs27cr** never truly engaged.
4. **Loss reason maps to stage → to a fix:** competitor/pricing losses early; **financing-not-approved** clusters at test-drive/negotiation.
5. **Channel quality:** walk-ins deliver ~half of all revenue at 45.7% conversion; social media is the weakest channel by both conversion and revenue/lead.
6. **Actionable now (as of 2025-12-31):** 35 cold leads, 32 stale order-placed (~Rs7.6cr), 30 past expected close (~Rs6.8cr).
7. **Anomalies handled:** 14 malformed 'lost' records surfaced as 'Unknown'.
8. **Deliberately NOT built:** forecasting (pipeline too small vs targets), speed-to-lead & touch-count predictors (no real / circular signal).